In [8]:
import pandas as pd
import plotly.express as px
import numpy as np


In [10]:
sales_df=pd.read_csv(r"F:\KPIT\Pandas_All\Datasets\sales_data.csv",parse_dates=["date"])
sales_df["category"]=pd.Categorical(sales_df["category"])
print(sales_df.head(4))
# print(type(sales_df["date"]))

        date region    store     category     product   sales  quantity  \
0 2025-01-01  North  Store_A  Electronics      Laptop  120000        15   
1 2025-01-01  North  Store_A     Clothing       Shirt   35000        70   
2 2025-01-01  South  Store_B  Electronics  Smartphone   95000        25   
3 2025-01-02   East  Store_C    Furniture       Chair   22000        40   

   discount  
0        10  
1         5  
2        12  
3         8  


In [11]:
print(sales_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   date      21 non-null     datetime64[ns]
 1   region    21 non-null     object        
 2   store     21 non-null     object        
 3   category  21 non-null     category      
 4   product   21 non-null     object        
 5   sales     21 non-null     int64         
 6   quantity  21 non-null     int64         
 7   discount  21 non-null     int64         
dtypes: category(1), datetime64[ns](1), int64(3), object(3)
memory usage: 1.4+ KB
None


### Load the dataset and retrieve all rows where the region is "North" and category is "Electronics" with sales greater than 1,00,000.

In [12]:
filtered_data=sales_df.loc[(sales_df["region"]=="North") & (sales_df["category"]=="Electronics") & (sales_df["sales"] > 100000)]
filtered_data

,date,region,store,category,product,sales,quantity,discount
0,2025-01-01,North,Store_A,Electronics,Laptop,120000,15,10
9,2025-01-04,North,Store_A,Electronics,Laptop,130000,17,8


### Find total sales and total quantity sold for each region. Sort them in descending order of sales.

In [14]:
grouped_data=sales_df.groupby("region")[["sales","quantity"]].sum()
grouped_data.sort_values(by="sales",ascending=False)

,sales,quantity
region,,
North,457000,169
South,406000,86
West,183000,250
East,170000,168


### Find which product recorded the highest single-day sales in the entire dataset.

In [18]:
single_product=sales_df.loc[[sales_df["sales"].idxmax()]]
single_product


,date,region,store,category,product,sales,quantity,discount
9,2025-01-04,North,Store_A,Electronics,Laptop,130000,17,8


### Create a pivot table showing total sales per region and category.

In [23]:
# pivot=sales_df.pivot_table(index=)
gr_data=sales_df.groupby(["region","category"])[["sales"]].sum()
gr_data.pivot_table(index="region",values="sales",columns="category",aggfunc="sum")

C:\Users\Pragyan Prakhar\AppData\Local\Temp\ipykernel_12676\2154782722.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  gr_data=sales_df.groupby(["region","category"])[["sales"]].sum()
C:\Users\Pragyan Prakhar\AppData\Local\Temp\ipykernel_12676\2154782722.py:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  gr_data.pivot_table(index="region",values="sales",columns="category",aggfunc="sum")


category,Clothing,Electronics,Furniture
region,,,
East,33000,115000,22000
North,95000,362000,0
South,0,194000,212000
West,123000,0,60000


### Convert the date column to datetime, set it as index, and resample the data to weekly frequency. Find total sales per week.

In [26]:
new_data=sales_df.set_index("date")
new_data.resample("W")[["sales"]].agg("sum")

,sales
date,
2025-01-05,720000
2025-01-12,496000


### First, create a pivot table showing total sales for each region (index) and category (columns). Then, melt it back to a long format to get columns: region, category, total_sales.

In [38]:
gr_data=sales_df.groupby(["region","category"])[["sales"]].sum()
gr_data=gr_data.pivot_table(index="region",values="sales",columns="category",aggfunc="sum")
display(gr_data)
gr_data=gr_data.reset_index()
gr_data.index
pd.melt(gr_data,id_vars=["region"],var_name="category" , value_name="total_sales")

C:\Users\Pragyan Prakhar\AppData\Local\Temp\ipykernel_12676\2285743896.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  gr_data=sales_df.groupby(["region","category"])[["sales"]].sum()
C:\Users\Pragyan Prakhar\AppData\Local\Temp\ipykernel_12676\2285743896.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  gr_data=gr_data.pivot_table(index="region",values="sales",columns="category",aggfunc="sum")


category,Clothing,Electronics,Furniture
region,,,
East,33000,115000,22000
North,95000,362000,0
South,0,194000,212000
West,123000,0,60000


,region,category,total_sales
0,East,Clothing,33000
1,North,Clothing,95000
2,South,Clothing,0
3,West,Clothing,123000
4,East,Electronics,115000
5,North,Electronics,362000
6,South,Electronics,194000
7,West,Electronics,0
8,East,Furniture,22000
9,North,Furniture,0


### For each region, calculate the day-to-day change in sales (current_day_sales - previous_day_sales).

In [60]:
temp_df=sales_df.copy()
temp_df=temp_df.sort_values(by=["region","date"]).reset_index(drop=True)
display(temp_df.index)
temp_df['prev_sale_day']=temp_df.groupby('region')['sales'].shift(1)  # Each region is treated separately
temp_df['daily_change'] = temp_df['sales'] - temp_df['prev_sale_day']
temp_df

RangeIndex(start=0, stop=21, step=1)

,date,region,store,category,product,sales,quantity,discount,prev_sale_day,daily_change
0,2025-01-02,East,Store_C,Furniture,Chair,22000,40,8,NaN,NaN
1,2025-01-03,East,Store_C,Electronics,Smartwatch,40000,18,6,22000.0,18000.0
2,2025-01-05,East,Store_C,Electronics,Headphones,30000,30,10,40000.0,-10000.0
3,2025-01-07,East,Store_C,Clothing,Shirt,33000,60,8,30000.0,3000.0
4,2025-01-09,East,Store_C,Electronics,Smartwatch,45000,20,6,33000.0,12000.0
5,2025-01-01,North,Store_A,Electronics,Laptop,120000,15,10,NaN,NaN
6,2025-01-01,North,Store_A,Clothing,Shirt,35000,70,5,120000.0,-85000.0
7,2025-01-03,North,Store_A,Electronics,Tablet,55000,20,9,35000.0,20000.0
8,2025-01-04,North,Store_A,Electronics,Laptop,130000,17,8,55000.0,75000.0
9,2025-01-06,North,Store_A,Clothing,Jacket,60000,25,10,130000.0,-70000.0


### Group the data by both region and category, and calculate:
### total sales (sum)
### average discount (mean)
### total quantity (sum)

In [55]:
grp_data=sales_df.groupby(["region","category"])[["sales","discount","quantity"]].agg(
    total_sales = ("sales","sum"),
    average_discount=("discount","mean"),
    total_quantity=("quantity","sum")
)
grp_data

C:\Users\Pragyan Prakhar\AppData\Local\Temp\ipykernel_12676\2724599841.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grp_data=sales_df.groupby(["region","category"])[["sales","discount","quantity"]].agg(


total_sales  average_discount  total_quantity
region category                                                  
East   Clothing           33000          8.000000              60
       Electronics       115000          7.333333              68
       Furniture          22000          8.000000              40
North  Clothing           95000          7.500000              95
       Electronics       362000          8.500000              74
       Furniture              0               NaN               0
South  Clothing               0               NaN               0
       Electronics       194000         12.000000              53
       Furniture         212000         13.000000              33
West   Clothing          123000          5.666667             180
       Electronics            0               NaN               0
       Furniture          60000          7.500000              70

In [57]:
px.line(sales_df,x='date',y='sales',color="region",markers=True)